# Run the Stage 1 GHI pipeline (orchestrator)

[!] writing to `*_v2` tables so his originals are never touched. [!] 

```
build_structured_data  →  build_split_days  →  build_ghi_model  →  build_all_uncurtailedpv
        (_v2)                   (_v2)                (_v2)                  (_v2)
```
**Cost note:** the structured_data build scans `ts` (billions of rows). Run the
**tiny test slice first** (one month, one site-part) and confirm it works before
scaling to the full year. Athena bills by data scanned.

In [1]:
# Bootstrap: paths + imports
import sys, pathlib
SHARED = pathlib.Path(r"../../shared")
STAGE1  = pathlib.Path(r"../stage1_ghi_pipeline")
sys.path.insert(0, str(SHARED))
sys.path.insert(0, str(STAGE1))

from aws_config import aq            # your existing Athena helper
from ciccada_config import SAI       # 'solar_analytics_iceberg'

import build_structured_data   as b1
import build_split_days        as b2
import build_ghi_model         as b3
import build_mape_quality_gate as b3b
import build_all_uncurtailedpv as b4

print("Target tables (note the _v2 suffix. Original results untouched):")
print(" ", b1.TARGET)
print(" ", b2.TARGET)
print(" ", b3.TARGET)
print(" ", b4.TARGET)

Target tables (note the _v2 suffix. Original results untouched):
  structured_data_v2
  split_days_v2
  pv_ghi_norm_model_v2
  all_uncurtailedpv_v2


## Step 1. Structured_data_v2

In [2]:
# 1a. Create the empty table (safe: drops & recreates only the _v2 table)
print(b1.create_table(aq, database=SAI))

Created empty structured_data_v2


In [3]:
# 1b. TEST SLICE FIRST. 
# One month, one of 8 site-parts.
# Confirm this completes and validate() looks sane BEFORE the full run.
b1.run_slice(aq, database=SAI, year=2024, months=[1], n_parts=8, parts=[0])

loaded year=2024 month=1 part=0/8


['loaded year=2024 month=1 part=0/8']

In [4]:
# 1c. Validate the test slice
b1.validate(aq, database=SAI)

Distinct sites: 1,165
Voltage / P_norm sanity:
 v_min  v_avg  v_max  p_norm_avg
  22.3  241.3  262.6       0.369


(      n
 0  1165,
    v_min  v_avg  v_max  p_norm_avg
 0   22.3  241.3  262.6       0.369)

In [5]:
b1.run_slice(aq, database=SAI, year=2024, months=[1], n_parts=16, parts=[0])

loaded year=2024 month=1 part=0/16


['loaded year=2024 month=1 part=0/16']

In [6]:
# 1d. FULL RUN. All months, all 16 site-parts, 2024 + 2025.
#     Uses run_resilient: paces the queries, retries on "exhausted resources",
#     and halves any slice that still fails. A failed Athena INSERT writes
#     nothing, so retries and sub-splits cannot duplicate rows.
#     create_table first: the two failed runs left partial data behind.
print(b1.create_table(aq, database=SAI))

Created empty structured_data_v2


In [7]:
done24, failed24 = b1.run_resilient(b1, aq, SAI, year=2024, months=range(1, 13), n_parts=8, pause=1)

loaded year=2024 month=1 part=0/8
loaded year=2024 month=1 part=1/8
loaded year=2024 month=1 part=2/8
loaded year=2024 month=1 part=3/8
loaded year=2024 month=1 part=4/8
loaded year=2024 month=1 part=5/8
loaded year=2024 month=1 part=6/8
loaded year=2024 month=1 part=7/8
loaded year=2024 month=2 part=0/8
loaded year=2024 month=2 part=1/8
loaded year=2024 month=2 part=2/8
loaded year=2024 month=2 part=3/8
loaded year=2024 month=2 part=4/8
loaded year=2024 month=2 part=5/8
loaded year=2024 month=2 part=6/8
loaded year=2024 month=2 part=7/8
loaded year=2024 month=3 part=0/8
loaded year=2024 month=3 part=1/8
loaded year=2024 month=3 part=2/8
loaded year=2024 month=3 part=3/8
loaded year=2024 month=3 part=4/8
loaded year=2024 month=3 part=5/8
loaded year=2024 month=3 part=6/8
loaded year=2024 month=3 part=7/8
loaded year=2024 month=4 part=0/8
loaded year=2024 month=4 part=1/8
loaded year=2024 month=4 part=2/8
loaded year=2024 month=4 part=3/8
loaded year=2024 month=4 part=4/8
loaded year=20

In [8]:
done25, failed25 = b1.run_resilient(b1, aq, SAI, year=2025, months=range(1, 13), n_parts=8, pause=1)

loaded year=2025 month=1 part=0/8
loaded year=2025 month=1 part=1/8
loaded year=2025 month=1 part=2/8
loaded year=2025 month=1 part=3/8
loaded year=2025 month=1 part=4/8
loaded year=2025 month=1 part=5/8
loaded year=2025 month=1 part=6/8
loaded year=2025 month=1 part=7/8
loaded year=2025 month=2 part=0/8
loaded year=2025 month=2 part=1/8
loaded year=2025 month=2 part=2/8
loaded year=2025 month=2 part=3/8
loaded year=2025 month=2 part=4/8
loaded year=2025 month=2 part=5/8
loaded year=2025 month=2 part=6/8
loaded year=2025 month=2 part=7/8
loaded year=2025 month=3 part=0/8
loaded year=2025 month=3 part=1/8
loaded year=2025 month=3 part=2/8
loaded year=2025 month=3 part=3/8
loaded year=2025 month=3 part=4/8
loaded year=2025 month=3 part=5/8
loaded year=2025 month=3 part=6/8
loaded year=2025 month=3 part=7/8
loaded year=2025 month=4 part=0/8
loaded year=2025 month=4 part=1/8
loaded year=2025 month=4 part=2/8
loaded year=2025 month=4 part=3/8
loaded year=2025 month=4 part=4/8
loaded year=20

In [9]:
b1.validate(aq, database=SAI)

Distinct sites: 15,454
Voltage / P_norm sanity:
 v_min  v_avg  v_max  p_norm_avg
   0.1  241.7  300.0       0.363


(       n
 0  15454,
    v_min  v_avg  v_max  p_norm_avg
 0    0.1  241.7  300.0       0.363)

In [10]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM structured_data_v2
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=SAI)

,n_duplicate_keys
0,0


## Step 2. split_days_v2

In [11]:
print(b2.create_table(aq, database=SAI))
print(b2.run(aq, database=SAI))
b2.validate(aq, database=SAI)

Created empty split_days_v2
Populated split_days_v2
Train / val split:
day_type  n_site_days  n_sites
   train      4984884    15444
     val      1253881    15445


,day_type,n_site_days,n_sites
0,train,4984884,15444
1,val,1253881,15445


## Step 3. pv_ghi_norm_model_v2

In [12]:
print(b3.create_table(aq, database=SAI))
b3.run(aq, database=SAI, years=(2024, 2025))   # ONE call, both years
b3.validate(aq, database=SAI)

Created empty pv_ghi_norm_model_v2
fitted years=2024, 2025 part=0/1
Model rows: 1,974,126  across 15,443 sites
Duplicate (site_id, tod_bin) keys (MUST be 0): 0
Sample fits (expect a+b near 1.0):
 site_id      tod_bin      a     b   n
  130538 05:35:00.000 -0.097 1.097  30
  130538 05:40:00.000  0.298 0.702  41
  130538 05:45:00.000  0.146 0.854  46
  130538 05:50:00.000  0.252 0.748  50
  130538 05:55:00.000  0.197 0.803  63
  130538 06:00:00.000  0.369 0.631  81
  130538 06:05:00.000  0.295 0.705 107
  130538 06:10:00.000  0.376 0.624 128


(   n_sites   n_rows
 0    15443  1974126,
    n
 0  0,
    site_id       tod_bin      a      b    n
 0   130538  05:35:00.000 -0.097  1.097   30
 1   130538  05:40:00.000  0.298  0.702   41
 2   130538  05:45:00.000  0.146  0.854   46
 3   130538  05:50:00.000  0.252  0.748   50
 4   130538  05:55:00.000  0.197  0.803   63
 5   130538  06:00:00.000  0.369  0.631   81
 6   130538  06:05:00.000  0.295  0.705  107
 7   130538  06:10:00.000  0.376  0.624  128)

In [13]:
# Step 3b. MAPE quality gate, now saving an auditable CSV
MAPE_CSV = "mape_under50_sites.csv"
mape_df, good_sites = b3b.run(aq, database=SAI, csv_path=MAPE_CSV)

Total sites with validation data: 15,380
MAPE distribution:
count    15380.0
mean        36.1
std         13.0
min          0.4
25%         29.7
50%         34.6
75%         40.2
max        434.1
Name: mape_pct, dtype: float64

Sites with MAPE < 50%: 14,299 (93.0% of total)
Saved to mape_under50_sites.csv


## Step 4. all_uncurtailedpv_v2

In [2]:
MAPE_CSV = r"mape_under50_sites.csv" 
print(b4.create_table(aq, database=SAI))
b4.run_year(aq, database=SAI, year=2024, mape_csv_path=MAPE_CSV, n_parts=6)
b4.run_year(aq, database=SAI, year=2025, mape_csv_path=MAPE_CSV, n_parts=6)
b4.validate(aq, database=SAI)

Created empty all_uncurtailedpv_v2
applied year=2024 part=0/6
applied year=2024 part=1/6
applied year=2024 part=2/6
applied year=2024 part=3/6
applied year=2024 part=4/6
applied year=2024 part=5/6
applied year=2025 part=0/6
applied year=2025 part=1/6
applied year=2025 part=2/6
applied year=2025 part=3/6
applied year=2025 part=4/6
applied year=2025 part=5/6
Rows with uncurtailed_P < P_kw (should be 0): 0
R6 nameplate cap impact:
   n_rows  n_capped  pct_capped
467332679    554072       0.119


(   n
 0  0,
       n_rows  n_capped  pct_capped
 0  467332679    554072       0.119)

## Done. Stage 1 rebuilt

# Optional comparisons with original

## Optional 1: Report on validation metrics

In [3]:
# structured data built
b1.validate(aq, database=SAI)

Distinct sites: 15,454
Voltage / P_norm sanity:
 v_min  v_avg  v_max  p_norm_avg
   0.1  241.7  300.0       0.363


(       n
 0  15454,
    v_min  v_avg  v_max  p_norm_avg
 0    0.1  241.7  300.0       0.363)

In [4]:
# build_split_days
b2.validate(aq, database=SAI)

Train / val split:
day_type  n_site_days  n_sites
   train      4984884    15444
     val      1253881    15445


,day_type,n_site_days,n_sites
0,train,4984884,15444
1,val,1253881,15445


In [5]:
# GHI model
b3.validate(aq, database=SAI)

Model rows: 1,974,126  across 15,443 sites
Duplicate (site_id, tod_bin) keys (MUST be 0): 0
Sample fits (expect a+b near 1.0):
 site_id      tod_bin      a     b   n
  130538 05:35:00.000 -0.097 1.097  30
  130538 05:40:00.000  0.298 0.702  41
  130538 05:45:00.000  0.146 0.854  46
  130538 05:50:00.000  0.252 0.748  50
  130538 05:55:00.000  0.197 0.803  63
  130538 06:00:00.000  0.369 0.631  81
  130538 06:05:00.000  0.295 0.705 107
  130538 06:10:00.000  0.376 0.624 128


(   n_sites   n_rows
 0    15443  1974126,
    n
 0  0,
    site_id       tod_bin      a      b    n
 0   130538  05:35:00.000 -0.097  1.097   30
 1   130538  05:40:00.000  0.298  0.702   41
 2   130538  05:45:00.000  0.146  0.854   46
 3   130538  05:50:00.000  0.252  0.748   50
 4   130538  05:55:00.000  0.197  0.803   63
 5   130538  06:00:00.000  0.369  0.631   81
 6   130538  06:05:00.000  0.295  0.705  107
 7   130538  06:10:00.000  0.376  0.624  128)

In [6]:
# Uncurtailed PV
b4.validate(aq, database=SAI)

Rows with uncurtailed_P < P_kw (should be 0): 0
R6 nameplate cap impact:
   n_rows  n_capped  pct_capped
467332679    554072       0.119


(   n
 0  0,
       n_rows  n_capped  pct_capped
 0  467332679    554072       0.119)

In [7]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM all_uncurtailedpv_v2
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=SAI)

,n_duplicate_keys
0,0


In [8]:
aq("""
    SELECT count(*) AS n_sites_with_multiple_caps
    FROM (
        SELECT site_id, count(DISTINCT ac_capacity_kw) AS n
        FROM meta_up23c
        WHERE is_pv = True
        GROUP BY site_id
        HAVING count(DISTINCT ac_capacity_kw) > 1
    )
""", database=SAI)

,n_sites_with_multiple_caps
0,0


In [9]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, tod_bin
        FROM pv_ghi_norm_model_v2
        GROUP BY site_id, tod_bin
        HAVING count(*) > 1
    )
""", database=SAI)

,n_duplicate_keys
0,0


## Optional 2: Comapre with V1 (original results included on the Milestone 3 report)

In [10]:
# Comparison: _v2 vs originals
compare = aq(f"""
    SELECT 
        'original' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(min(V), 1) AS v_min,
        round(avg(V), 1) AS v_avg,
        round(max(V), 1) AS v_max,
        round(avg(P_kw_norm), 4) AS p_norm_avg
    FROM structured_data
    UNION ALL
    SELECT
        'v2' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(min(V), 1) AS v_min,
        round(avg(V), 1) AS v_avg,
        round(max(V), 1) AS v_max,
        round(avg(P_kw_norm), 4) AS p_norm_avg
    FROM structured_data_v2
""", database=SAI)
print("structured_data: original vs v2")
print(compare.to_string(index=False))




structured_data: original vs v2
 version     n_rows  n_sites  v_min  v_avg  v_max  p_norm_avg
original 1022900647    15454    0.1  241.2  293.5      0.3606
      v2  841491009    15454    0.1  241.7  300.0      0.3625


In [11]:
#  What to expect: 
# Site counts should be similar but not identica
# _v2 should have slightly fewer sites because of [flex_export exclusion, ~539 sites removed]. 
# The v_max should be higher in _v2 because of the switch from avg to max voltage. 
# The max_uncurt should be lower in _v2 because now it caps at nameplate.

compare_unc = aq(f"""
    SELECT
        'original' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(avg(uncurtailed_P), 2) AS avg_uncurt,
        round(max(uncurtailed_P), 2) AS max_uncurt
    FROM all_uncurtailedpv
    UNION ALL
    SELECT
        'v2' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(avg(uncurtailed_P), 2) AS avg_uncurt,
        round(max(uncurtailed_P), 2) AS max_uncurt
    FROM all_uncurtailedpv_v2
""", database=SAI)
print("\nall_uncurtailedpv: original vs v2")
print(compare_unc.to_string(index=False))


all_uncurtailedpv: original vs v2
 version     n_rows  n_sites  avg_uncurt  max_uncurt
      v2  467332679    14299        5.43      334.83
original 1117984493    15261        5.39     1349.49


In [12]:
funnel = aq(f"""
    SELECT 'structured_data_v2' AS stage, count(DISTINCT site_id) AS n_sites
    FROM structured_data_v2
    UNION ALL
    SELECT 'split_days_v2', count(DISTINCT site_id) FROM split_days_v2
    UNION ALL
    SELECT 'ghi_model_v2', count(DISTINCT site_id) FROM pv_ghi_norm_model_v2
    UNION ALL
    SELECT 'mape_csv', count(DISTINCT site_id) FROM all_uncurtailedpv_v2
""", database=SAI)
print("\nSite funnel through Stage 1:")
print(funnel.to_string(index=False))


Site funnel through Stage 1:
             stage  n_sites
     split_days_v2    15445
      ghi_model_v2    15443
          mape_csv    14299
structured_data_v2    15454


In [13]:
coverage = aq(f"""
    SELECT year, month, count(*) AS n_rows, count(DISTINCT site_id) AS n_sites
    FROM structured_data_v2
    GROUP BY year, month
    ORDER BY year, month
""", database=SAI)
print("\nMonthly coverage:")
print(coverage.to_string(index=False))


Monthly coverage:
 year  month   n_rows  n_sites
 2024      1 41152818     9396
 2024      2 36444234    10308
 2024      3 37642356    10481
 2024      4 39313017    11390
 2024      5 34774494    11268
 2024      6 32467712    11334
 2024      7 33141404    10970
 2024      8 33677741    10779
 2024      9 33312294    10719
 2024     10 43266734    10867
 2024     11 39293833     9893
 2024     12 49938955    10963
 2025      1 35580683     8364
 2025      2 36002027    10090
 2025      3 33022870     9926
 2025      4 36125466    10255
 2025      5 31373827    10254
 2025      6 17270832    10080
 2025      7 26826871     9885
 2025      8 33705780     9941
 2025      9 30776104     9601
 2025     10 32451337     9522
 2025     11 32353798     8589
 2025     12 41575822     8916


In [14]:
# What to expect: _v2 should show higher average voltage and a higher percentage above 240V. 
# Vmax catches the high phase that avg was hiding. 
# The difference quantifies how much conformance was being undercounted.


r1_impact = aq(f"""
    SELECT
        round(avg(V), 2) AS v2_avg_of_max,
        round(count(CASE WHEN V > 240 THEN 1 END) * 100.0 / count(*), 2)
            AS pct_above_240_v2
    FROM structured_data_v2
    WHERE P_kw_norm > 0.05
""", database=SAI)

r1_original = aq(f"""
    SELECT
        round(avg(V), 2) AS orig_avg_of_avg,
        round(count(CASE WHEN V > 240 THEN 1 END) * 100.0 / count(*), 2)
            AS pct_above_240_orig
    FROM structured_data
    WHERE P_kw_norm > 0.05
""", database=SAI)

print("\nFix impact (avg→max voltage):")
print(f"  Original avg(voltage):  mean={r1_original['orig_avg_of_avg'].iloc[0]}V, "
      f"{r1_original['pct_above_240_orig'].iloc[0]}% above 240V")
print(f"  v2 max(voltage):        mean={r1_impact['v2_avg_of_max'].iloc[0]}V, "
      f"{r1_impact['pct_above_240_v2'].iloc[0]}% above 240V")


Fix impact (avg→max voltage):
  Original avg(voltage):  mean=241.57V, 65.04% above 240V
  v2 max(voltage):        mean=242.09V, 68.94% above 240V


In [15]:
r2_impact = aq(f"""
    SELECT
        count(DISTINCT o.site_id) AS in_original_only
    FROM (SELECT DISTINCT site_id FROM structured_data) o
    LEFT JOIN (SELECT DISTINCT site_id FROM structured_data_v2) v
        ON o.site_id = v.site_id
    WHERE v.site_id IS NULL
""", database=SAI)
print(f"\n[Flex export] impact: {int(r2_impact['in_original_only'].iloc[0])} sites in original "
      f"but excluded from v2 (flex_export + other filter differences)")


[Flex export] impact: 0 sites in original but excluded from v2 (flex_export + other filter differences)


In [16]:
by_state = aq(f"""
    SELECT m.state, count(DISTINCT sd.site_id) AS n_sites
    FROM structured_data_v2 sd
    JOIN (SELECT DISTINCT site_id, state FROM meta_up23c) m ON sd.site_id = m.site_id
    GROUP BY m.state
    ORDER BY n_sites DESC
""", database=SAI)
print("\nSites by state:")
print(by_state.to_string(index=False))


Sites by state:
state  n_sites
  NSW     6003
  QLD     5764
  VIC     1306
   SA     1229
   WA      485
  TAS      361
  ACT      188
   NT      118
